In [1]:
from embedder import Embedder

embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print("Shape:", v.shape)      
print("First value v[0]:", v[0])

Shape: (384,)
First value v[0]: -0.02058203437252893


In [2]:
import requests

url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/8c1834d/02-vector-search/lessons/07-sqlitesearch-vector.md"
doc_text = requests.get(url).text

d = embedder.encode(doc_text)

similarity = v.dot(d)

print("Cosine similarity:", similarity)

Cosine similarity: 0.36107027225589694


In [3]:
from gitsource import GithubRepositoryDataReader, chunk_documents
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(documents)} documents")

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks: {len(chunks)}")
print(chunks[0])

texts = [chunk['content'] for chunk in chunks]
X = embedder.encode_batch(texts)
print(f"Embeddings shape: {X.shape}")

scores = X.dot(v)
best_index = scores.argmax()
print("Best chunk filename:", chunks[best_index]['filename'])

Loaded 72 documents
Total chunks: 295
{'start': 0, 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the

In [4]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

q4_query = "What metric do we use to evaluate a search engine?"
q4_vector = embedder.encode(q4_query)

results = vindex.search(q4_vector, num_results=5)
print("Top result filename:", results[0]["filename"])
for i, r in enumerate(results):
    print(f"  {i+1}. {r['filename']}")

Top result filename: 04-evaluation/lessons/05-search-metrics.md
  1. 04-evaluation/lessons/05-search-metrics.md
  2. 04-evaluation/lessons/01-intro.md
  3. 01-agentic-rag/lessons/05-search.md
  4. 04-evaluation/lessons/01-intro.md
  5. 04-evaluation/lessons/15-next-steps.md


In [5]:
import minsearch

tindex = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
tindex.fit(chunks)

q5_query = "How do I store vectors in PostgreSQL?"
text_results = tindex.search(q5_query, num_results=5)

q5_vector = embedder.encode(q5_query)
vector_results = vindex.search(q5_vector, num_results=5)

text_files = {r["filename"] for r in text_results}
vector_files = {r["filename"] for r in vector_results}

print("Text results:")
for r in text_results:
    print(" ", r["filename"])

print("\nVector results:")
for r in vector_results:
    print(" ", r["filename"])

print("\nIn vector only:", vector_files - text_files)

Text results:
  02-vector-search/lessons/02-embeddings.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md

Vector results:
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md

In vector only: {'02-vector-search/lessons/08-pgvector.md'}


In [6]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


q6_query = "How do I give the model access to tools?"


text_results = tindex.search(q6_query, num_results=5)

q6_vector = embedder.encode(q6_query)
vector_results = vindex.search(q6_vector, num_results=5)

hybrid_results = rrf([text_results, vector_results])

print("Top result:", hybrid_results[0]["filename"])
for i, r in enumerate(hybrid_results):
    print(f"  {i+1}. {r['filename']}")


Top result: 01-agentic-rag/lessons/13-function-calling.md
  1. 01-agentic-rag/lessons/13-function-calling.md
  2. 01-agentic-rag/lessons/14-agentic-loop.md
  3. 01-agentic-rag/lessons/01-intro.md
  4. 04-evaluation/lessons/02-ground-truth.md
  5. 01-agentic-rag/lessons/13-function-calling.md
